## 1. 라이브러리와 오류 범주

In [ ]:
from __future__ import annotations
import csv
from pathlib import Path

CATEGORIES = (
    "legal_terminology", "long_document", "multiple_labels",
    "proper_nouns_institutions", "numbers_dates_articles", "translation_expression"
)

## 2. 코딩 시트 읽기·Binary Kappa

In [ ]:
def get_rows(path):
    with Path(path).open(encoding='utf-8-sig',newline='') as f:
        rows=list(csv.DictReader(f))
    if len(rows)!=40 or len({r['case_id'] for r in rows})!=40:
        raise ValueError(f'{path}: expected 40 unique case IDs')
    return {r['case_id']:r for r in rows}

def binary_kappa(a,b):
    if len(a)!=len(b) or not a:raise ValueError('nonempty equal length arrays required')
    if not set(a+b).issubset({0,1}):raise ValueError('binary codes 0/1 required')
    n=len(a);observed=sum(x==z for x,z in zip(a,b))/n
    pa=sum(a)/n; pb=sum(b)/n
    expected=pa*pb+(1-pa)*(1-pb)
    return observed, ((observed-expected)/(1-expected) if abs(1-expected)>1e-12 else None)

## 3. 전체 / FI / FR별 여섯 범주의 결과표 생성

In [ ]:
def run(dfile,efile,outfile):
    d=get_rows(dfile);e=get_rows(efile)
    if set(d)!=set(e):raise ValueError('D/E case_id sets differ')
    results=[]
    for language,subset in [('ALL',sorted(d)),('FI',sorted(i for i in d if i.startswith('FI_'))),
                            ('FR',sorted(i for i in d if i.startswith('FR_')))]:
        for category in CATEGORIES:
            ac=[];bc=[]
            for key in subset:
                a=d[key][category].strip();b=e[key][category].strip()
                if a not in ('0','1') or b not in ('0','1'):
                    raise ValueError(f'Uncoded/invalid value in {key}, {category}: D={a!r}, E={b!r}')
                ac.append(int(a));bc.append(int(b))
            observed,kappa=binary_kappa(ac,bc)
            results.append({'language':language,'category':category,'n_cases':len(subset),
                 'agreement_pct':round(observed*100,4),
                 'cohens_kappa':round(kappa,6) if kappa is not None else '',
                 'kappa_status':'defined' if kappa is not None else 'undefined_expected_agreement_1',
                 'coder_D_positive':sum(ac),'coder_E_positive':sum(bc)})
    outfile=Path(outfile);outfile.parent.mkdir(exist_ok=True,parents=True)
    with outfile.open('w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(results[0]));w.writeheader();w.writerows(results)
    return results

## 4. 합성 데이터 단위 테스트

아래 테스트는 실제 프로젝트 코딩 결과를 사용하지 않습니다.

In [ ]:
def test():
    # Unit tests only; these are synthetic vectors, never actual project annotation results.
    assert binary_kappa([0,0,1,1],[0,0,1,1])==(1.0,1.0)
    po,k=binary_kappa([0,0,1,1],[0,1,0,1]);assert abs(po-.5)<1e-9 and abs(k)<1e-9
    po,k=binary_kappa([0,0,0,0],[0,0,0,0]);assert po==1 and k is None
    try:run('/should/not/exist','/should/not/exist','/tmp/no.csv')
    except FileNotFoundError:pass
    else:raise AssertionError('No rows should not silently pass')
    print('PASS: synthetic Cohen kappa unit tests; no actual annotations used')

In [ ]:
test()

## 5. 실제 D/E 코딩 결과로 실행

두 사람이 모든 항목을 작성한 뒤 경로를 지정하고 `RUN_KAPPA=True`로 바꾸세요. 미작성된 셀이 있으면 명시적인 오류를 내도록 설계했습니다.

In [ ]:
D_CODING_FILE = Path("coding_D.csv")
E_CODING_FILE = Path("coding_E.csv")
KAPPA_OUT_FILE = Path("cohens_kappa.csv")
RUN_KAPPA = False    # D/E 40건×6범주 독립 코딩 완료 후 True

if RUN_KAPPA:
    rows = run(D_CODING_FILE, E_CODING_FILE, KAPPA_OUT_FILE)
    print(f"완료: {len(rows)}개 결과 → {KAPPA_OUT_FILE.resolve()}")
else:
    print("실제 D/E 코딩 결과가 필요합니다. RUN_KAPPA=False 상태에서는 저장하지 않습니다.")